# Laboratorio 22: Valoración de Opciones con QAE

Quantum Amplitude Estimation acelera la estimación de E[f(X)] de O(1/ε²) a O(1/ε) respecto a Monte Carlo.

**Prerequisitos:** Módulo 18 (Complejidad), Lab 07 (QPE), Lab 38 (avanzado).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
print('Entorno listo')

## 1. Modelo financiero: opción call europea

Precio = E[max(S_T - K, 0)] · e^{-rT}. Discretizamos S_T en N=8 escenarios.

In [ ]:
S0, K, r, sigma, T = 100.0, 100.0, 0.05, 0.20, 1.0
N = 8
S_vals = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*np.linspace(-2,2,N))
probs  = np.exp(-0.5*np.linspace(-2,2,N)**2); probs /= probs.sum()
payoffs = np.maximum(S_vals - K, 0)
price_mc = np.sum(probs * payoffs) * np.exp(-r*T)
print(f'Precio Monte Carlo exacto: {price_mc:.4f}')
for i,(s,p,pay) in enumerate(zip(S_vals,probs,payoffs)):
    print(f'  S{i}: S_T={s:.1f} P={p:.3f} Payoff={pay:.2f}')

## 2. Preparación del estado cuántico

Codificamos la distribución de probabilidad: |ψ⟩ = Σ √p_i |i⟩

In [ ]:
# Codificar distribucion en estado cuantico |psi> = sum sqrt(p_i)|i>
n_qubits = int(np.log2(N))
amplitudes = np.sqrt(probs); amplitudes /= np.linalg.norm(amplitudes)

qc_prep = QuantumCircuit(n_qubits)
qc_prep.initialize(amplitudes, range(n_qubits))

sv = Statevector(qc_prep)
probs_check = sv.probabilities()
print('Verificacion probabilidades (original vs cuantico):')
for i,(p1,p2) in enumerate(zip(probs, probs_check)):
    print(f'  Escenario {i}: {p1:.4f} vs {p2:.4f}  diff={abs(p1-p2):.2e}')

## 3. Oráculo del Payoff

El qubit ancilla se rota para que su probabilidad de medir |1⟩ sea exactamente E[f] (el valor esperado normalizado del payoff).

In [ ]:
# Oraculo del Payoff: ancilla rotada segun f_i = payoff_i/payoff_max
payoff_norm = payoffs / (payoffs.max() + 1e-10)

# Construir amplitudes del sistema completo (n_qubits + 1 ancilla)
full_amp = np.zeros(2**(n_qubits + 1))
for i in range(N):
    f = payoff_norm[i]; a = amplitudes[i]
    full_amp[i]         = a * np.sqrt(1 - f)   # ancilla = 0
    full_amp[i + 2**n_qubits] = a * np.sqrt(f)  # ancilla = 1

qc_qae = QuantumCircuit(n_qubits + 1)
qc_qae.initialize(full_amp, range(n_qubits + 1))

sv_full = Statevector(qc_qae)
probs_full = sv_full.probabilities()
prob_ancilla_1 = sum(probs_full[2**n_qubits:])
price_qae = prob_ancilla_1 * payoffs.max() * np.exp(-r*T)

print(f'Amplitud^2 ancilla=|1>: {prob_ancilla_1:.4f}')
print(f'Precio QAE:             {price_qae:.4f}')
print(f'Precio Monte Carlo:     {price_mc:.4f}')
print(f'Error absoluto:         {abs(price_qae-price_mc):.4f}')

## 4. Ventaja cuántica: complejidad de muestreo

QAE logra aceleración cuadrática sobre Monte Carlo. Para ε=1%, MC necesita ~10,000 muestras; QAE ~100 consultas.

In [ ]:
# Comparativa de complejidad: MC vs QAE
np.random.seed(0)
n_list = [10,50,100,500,1000,5000,10000]
mc_err, qae_theory = [], []
for n in n_list:
    ests = [np.mean(payoffs[np.random.choice(N,size=n,p=probs)])*np.exp(-r*T)
            for _ in range(200)]
    mc_err.append(np.std(ests))
    qae_theory.append(np.pi/(2*n))

fig, ax = plt.subplots(figsize=(8,5))
ax.loglog(n_list, mc_err,      'b-o', lw=2, label='Monte Carlo O(1/sqrt(n))')
ax.loglog(n_list, qae_theory,  'r--s',lw=2, label='QAE teorico O(1/n)')
ax.set_xlabel('Numero de muestras / consultas')
ax.set_ylabel('Error de estimacion'); ax.set_title('MC vs QAE')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.savefig('qae_vs_mc.png', dpi=100)
plt.show()
print(f'Para error 1%: MC necesita ~{int((price_mc*0.01)**-2)} muestras, QAE ~{int(np.pi/(2*price_mc*0.01))} consultas')